In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [28]:
df = pd.read_csv("database[1].csv")

In [30]:
num_fill_cols = [
    'Azimuthal Gap',
    'Horizontal Distance',
    'Root Mean Square',
    'Horizontal Error',
    'Depth Error',
    'Magnitude Error',
    'Magnitude Seismic Stations',
    'Depth Seismic Stations',
]

In [31]:
for c in num_fill_cols:
    df[c] = df[c].fillna(df[c].mean())

In [32]:
df['Magnitude Type'] = df['Magnitude Type'].fillna(df['Magnitude Type'].mode()[0])

In [33]:
dfCleaned = pd.get_dummies(
    df,
    columns=['Magnitude Type', 'Source', 'Type', 'Status',
             'Location Source', 'Magnitude Source'],
    drop_first=True
)

In [34]:
bool_cols = dfCleaned.select_dtypes(include=['bool']).columns
dfCleaned[bool_cols] = dfCleaned[bool_cols].astype(int)
dfCleaned = dfCleaned.drop(['ID', 'Date', 'Time'], axis=1)

In [35]:
dfCleaned = dfCleaned.drop(
    ['Depth Seismic Stations', 'Magnitude Seismic Stations', 'Magnitude Error'],
    axis=1
)

In [36]:
scale_cols = ['Latitude', 'Longitude', 'Depth', 'Azimuthal Gap',
              'Horizontal Distance', 'Root Mean Square', 'Horizontal Error',
              'Depth Error']
scaler = StandardScaler()
dfCleaned[scale_cols] = scaler.fit_transform(dfCleaned[scale_cols])

In [37]:
x = dfCleaned.drop('Magnitude', axis=1)
y = dfCleaned['Magnitude']
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [38]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [39]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
cv_rmse = np.sqrt(-cross_val_score(
    model, x, y, cv=5, scoring='neg_mean_squared_error'
)).mean()

In [40]:
print(f"Random Forest -> MAE: {mae:.6f}  RMSE: {rmse:.6f}  R²: {r2:.6f}  CV_RMSE: {cv_rmse:.6f}")

Random Forest -> MAE: 0.281927  RMSE: 0.395055  R²: 0.153855  CV_RMSE: 0.408265


In [41]:
dt_model = DecisionTreeRegressor(
    criterion='squared_error',
    splitter='best',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

In [42]:
dt_model.fit(x_train, y_train)
y_pred_dt = dt_model.predict(x_test)

In [43]:
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)
cv_rmse_dt = np.sqrt(-cross_val_score(
    dt_model, x, y, cv=5, scoring='neg_mean_squared_error'
)).mean()

In [44]:
print(f"Decision Tree -> MAE: {mae_dt:.6f}  RMSE: {rmse_dt:.6f}  R²: {r2_dt:.6f}  CV_RMSE: {cv_rmse_dt:.6f}")

Decision Tree -> MAE: 0.368599  RMSE: 0.532530  R²: -0.537508  CV_RMSE: 0.541970


In [45]:
svm_model = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
svm_model.fit(x_train, y_train)
y_pred_svm = svm_model.predict(x_test)

In [47]:
mae_svm = mean_absolute_error(y_test, y_pred_svm)
rmse_svm = np.sqrt(mean_squared_error(y_test, y_pred_svm))
r2_svm = r2_score(y_test, y_pred_svm)
cv_rmse_svm = np.sqrt(-cross_val_score(
    svm_model, x, y, cv=5, scoring='neg_mean_squared_error'
)).mean()

In [48]:
print(f"SVM -> MAE: {mae_svm:.6f}  RMSE: {rmse_svm:.6f}  R²: {r2_svm:.6f}  CV_RMSE: {cv_rmse_svm:.6f}")

SVM -> MAE: 0.265217  RMSE: 0.398833  R²: 0.137596  CV_RMSE: 0.406327
